In [1]:
# This is a transformer from scratch (mostly for fun)

In [2]:

#imports

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

import os
import sys
import random

import typing

torch.set_default_device('cuda')

In [3]:
# data preperation 
# importantly, I am chose to only look at individual characters (rather than tokens) to make the transformer far simpler to implement (no need for a tokenizer)
# Moreover, by focusing on only characters, each character can just be simply assigned an integer, making the code simpler
text = open("trainingdata.txt", "r").read() # should be simple plain text file

chars = sorted(list(set(text)))
string_to_output_ints = {character: i for i, character in enumerate(chars)}
int_to_output_string = {i: character for i, character in enumerate(chars)}

encode = lambda s: [string_to_output_ints[c] for c in s] # encoder that converts any string into int based on training data
decode = lambda ids: "".join([int_to_output_string[i] for i in ids]) # basically the opposite of the encoder

data = torch.tensor(encode(text), dtype=torch.long) # takes all the training data and coverts into into a tensor; uses a datatype of torch.long to hold a signed 64 bit integer (int64)


In [4]:
# Mock up of the actual problem
# since the problem itself is "next token prediction" with each token being an individual character
# each of the inputs must be an array of chars, while the output is a char
# essentially if x = [list of chars] = the input, then y = [list of chars + new_char] = the output
# a good mathematical representation for this can be P(x_(t+1) | x_1, ... , x_t)

In [5]:
# token embedding works by creating a mathematical space (a vector space) with n_dimensions
# this vector space then houses all the information of each of the chars (tokens)
# This therefore converts the dictionary, which is a simplification done on the code, into a 
# sound vector space that the model can train on
vocab_size = len(chars) # number of unique chars that are present in the text
d_model = 384 # the number of dimensions each vector can have to represent each char
batch_size = 64
max_iters = 5000
eval_interval = 200
learning_rate = 5e-4
eval_iters = 20
num_heads = 6
num_layers = 6
dropout = 0.1
temperature = 0.2
BLOCK_SIZE: int = 256 # max sequence/context length
device: str = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
token_embeddings = nn.Embedding(vocab_size, d_model) # creates the actual embedding space


# Input:
# B = 4 sequences
# T = 10 tokens per sequence
x = torch.randint(0, vocab_size, (4, 10))

cuda


In [6]:
# positional embedding helps the attention mechanism understand the location at which the token is located in
# This works with regular token embedding and adds positional information 
# (which will change the meaning of each of the tokens)
positional_embedding = nn.Embedding(BLOCK_SIZE, d_model) # creates the positional embedding space

tok = token_embeddings(x)

# passes into the positional embedding the length of the data input and the device to compute
pos = positional_embedding(
    torch.arange(x.shape[1], device=x.device)
)

h = tok + pos # this is the combined embedding that describes the text, it combines the positional embeddings and the token embeddings together

print(h)


tensor([[[ 1.6045e-01,  1.1535e+00, -1.4061e+00,  ...,  1.3050e-01,
           2.2593e+00, -1.5938e+00],
         [-2.4106e+00, -6.5843e-01, -1.7058e-01,  ..., -1.7489e+00,
          -1.3282e+00, -4.7772e-01],
         [-1.3595e+00, -1.1190e+00, -2.4277e-01,  ...,  8.1381e-01,
           2.2588e-01, -7.2328e-01],
         ...,
         [-2.1143e+00,  5.8786e-01,  1.7523e+00,  ...,  1.1775e-02,
          -1.6798e+00,  1.7680e+00],
         [-9.4481e-01,  9.5406e-01, -9.4202e-01,  ...,  1.3401e-02,
           3.3084e-01, -2.3362e-01],
         [ 1.1664e+00, -2.5130e+00,  1.8083e+00,  ...,  1.6213e+00,
          -1.0647e+00, -2.7360e+00]],

        [[-4.7852e-01,  1.2158e+00,  3.5057e-01,  ...,  7.7194e-02,
           3.1538e+00, -1.7344e+00],
         [-2.0954e+00, -2.3466e+00,  7.9607e-01,  ..., -3.0884e+00,
           5.5922e-01,  1.4127e+00],
         [-5.2267e-01,  2.7563e-01,  1.0637e+00,  ..., -3.3556e-01,
           1.6394e+00,  1.5253e+00],
         ...,
         [-5.5453e-02,  1

In [7]:
# Convert the entire dataset into integer token IDs
data = torch.tensor(
    encode(text),
    dtype=torch.long
)
print(data.shape)


# generic training and validation split.
# the standard is usually 90 percent 10 percent because on either ends of the extremes
# with a LARGE amount of data, or a small amount of data, you need to use as much as possible
# in training to maximize performance
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

torch.Size([74805])


In [8]:
# batch making
def get_batch(split_name):
    """
    Returns:

        x: [B, T]
        y: [B, T]

    y is x shifted one character into the future. (next prediction/data)
    """

    source = train_data if split_name == "train" else val_data

    # random starting positions for each sequence
    starts = torch.randint(
        0,
        len(source) - BLOCK_SIZE - 1,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + BLOCK_SIZE]
        for i in starts
    ])

    y = torch.stack([
        source[i + 1:i + BLOCK_SIZE + 1]
        for i in starts
    ])

    return x.to(device), y.to(device)

In [9]:
# SINGLE ATTENTION HEAD!!!!
class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

        # Lower-triangular causal mask
        #
        # Example for T = 4:
        #
        # 1 0 0 0
        # 1 1 0 0
        # 1 1 1 0
        # 1 1 1 1
        # just a fancy matrix for a pretty simple idea

        self.register_buffer(
            "tril", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
        )  # this is very interesting to analyse
        # the purpose of having these 'buffers' is that the model should not be able to look at the future words,
        # and perform calculations based on them. By making the future tokens 0, the model will
        # be unable to allow them to influence current tokens, keeping the attention mechanism
        # one-way. In attention, the past cannot affect the future, until the future occurs
        # then only is the future able to affect the past, and vice-versa. (stops cheating)
        # softmax also resets these back down to 0 after attention makes them -inf

        self.dropout = nn.Dropout(
            dropout
        )  # this is actually also very interesting to analyse
        # essentially, the purpose of Dropout is to prevent overfitting of the data
        # Each node has a probability of being deactivated during training,
        # which ends up decreasing overfitting, and increasing overall model quality
        # A good analogy to this is in the workplace
        # If there are 10 workers who specialize in their own specific tasks, overtime,
        # they are only capable of outputting the very specific things that they were
        # trained to do, which means that they won't be able to challenging or unique problems
        # however, if one of them is sick, and cannot work, than the others pick up
        # some of the info that the missing worker carried, which makes all of them more
        # free-thinking and creative, ultimately making it so that when the sick workers
        # is better, and returns, the other workers are more stimulated, and less hyper-focused
        # on their own specialization, making them more generalized (stopping overfitting)

    """_summary_ This does the forward pass of all the input data. 
    Essentially, when the data gets a linear transformation
    _param_ x = [B, T, C]
    q,k,v = [B, T, H] where H is the head dimension
    
    """

    def forward(self, x):
        B, T, C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        # computing attention scores
        # it mostly works from using 
        # q: [B, T, H]
        # k.transpose [B, H, T]

        scores = q @ k.transpose(-2, -1)
        
        # this scales scores by sqrt(head_size)
        # The reason for this is very obvious once mathematically explained
        # Essentially, the dot product of vectors is always capped at 1, 
        # however, adding vectors together can increase the magnitude
        # beyond what softmax can effectively control
        # therefore producing unreliable extreme values that prevent the transformer from working
        # The dot product variance is always 1, while the summation effect causes the dot product variation
        # to become it's magnitude
        # dividing sets the scores back down to 1, and therefore keeps the stddev at 1
        scores = scores / (k.shape[-1] ** 0.5)
        
        # applies the casual mask (to prevent it from cheating from the future)
        scores = scores.masked_fill(self.tril[:T, :T] ==0, float('-inf') ) # pyright: ignore[reportIndexIssue]
        
        # SOFTMAX TIME (sexy function ngl)
        # quick recap of softmax is that it tones done extreme values, setting them between 0 and 1
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        
        # this is a weighted combo of all the values
        # essentially multiples the weight matrix and value matrix (Weight * Value)
        out = weights @ v
        
        # [B, T, H]
        return out
        
    

In [10]:
# Multi Head Attention, which is an extension of the current Single Headed one


class MultiHeadedAttention(nn.Module):
    def __init__(self, num_heads, d_model):
        super().__init__()
        # this is extremely important as the num of heads must be a multiple of the model
        # as otherwise the matrix multipliction fails
        assert d_model % num_heads == 0

        # create each head using the single head class
        head_size = d_model // num_heads
        self.heads = nn.ModuleList(
            [Head(d_model=d_model, head_size=head_size) for i in range(num_heads)]
        )

        # after concatting the heads, the heads need to be projected back into the d_model,
        # in order to preserve matrix operatioons
        self.projection = nn.Linear(d_model, d_model)

        # as prev explained, strategic disabling of param based on probability (dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # since each attention head is one after another, they need to be run seperately
        # I beleive this is where many attention layer optimizations can and are made
        # since parallel computing this could make it substantially faster and less time intensive

        outputs = [head(x) for head in self.heads]
        
        # combines every one of the outputs into a single matrix for the output
        # if each output from each head was [B, T, head_size]
        # then out would be [B, T, d_model]
        # THIS IS WHERE THE ENTIRE MATRIX OPERATIONS SIMPLIFICATION COMES IN
        # It is very difficult to concat if they were different sizes or dimensions,
        # and would lead to messier code
        out = torch.cat(outputs, dim=-1)
        
        out = self.projection(out) 
        out = self.dropout(out)
        return out

In [11]:
# This is the FeedForward layer, which allows for changes to each individual token's understanding and information
# rather than simply being an average of all the sorrounding tokens (which attention will do)
# this instead increases the info that each individual token holds
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        # this is just the neural network, very sexy and compact
        # GeLU is being used rather than ReLU since GeLU is universally better
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model, bias=False),
            nn.GELU(
                approximate="tanh"
            ),  # since tanh is very close to the actual GeLU, it can be used, it's more performance efficient
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model, bias=False),
            nn.Dropout(dropout),
        )

    # very simple forward passing, since each token is only modifying itself, rather than others
    def forward(self, x):
        return self.net(x)

In [12]:
# Transformer block essentially functions as such:
#  attention -> MLP(FeedForwardLayer) -> residual connections -> LayerNorm


class Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.self_attention = MultiHeadedAttention(num_heads=num_heads, d_model=d_model)

        self.ffwd = FeedForward(d_model=d_model)
        self.layer_norm_1 = nn.RMSNorm(
            d_model
        )  # RMSNorm used for performance, LayerNorm is fine

        # by multiplying layer weights by a small amount, the rate at which
        # the model learns is dampened, essentially allowing for smaller adjustment
        # at late-stage
        self.scale_1 = nn.Parameter(torch.ones(d_model) * 1e-2)
        self.scale_2 = nn.Parameter(torch.ones(d_model) * 1e-2)

    # this is also in parallel to allow for slightly faster compute
    def forward(self, x):

        normed_x = self.layer_norm_1(x)

        x = x + self.scale_1 * self.self_attention(normed_x) + self.scale_2 * self.ffwd(normed_x)
        return x

In [ ]:
# putting everything together now
class SmallLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.positional_embeddings = nn.Embedding(BLOCK_SIZE, d_model)

        # all the transformer layers now

        self.blocks = nn.Sequential(
            *[Block(d_model=d_model, num_heads=num_heads) for i in range(num_layers)]
        )

        # final layer normalization super important for general capability
        self.final_layer_normalization = nn.RMSNorm(d_model)

        # this converts all the embeddings BACK into vocab definitions
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape

        tokenal_embeddings = self.token_embeddings(x)

        # [B, T, C]
        positions = torch.arange(T, device=x.device)

        # positions shape is [T]

        position_embeddings = self.positional_embeddings(positions)

        # [T,C]
        x = position_embeddings + tokenal_embeddings
        x = self.blocks(x)
        x = self.final_layer_normalization(x)
        vocab_logits = self.lm_head(x)

        # LOSS COMPUTATIONS!!!!!
        # This is genuinly just finding a secant line
        loss = None
        if targets is not None:
            B, T, C = vocab_logits.shape

            # mostly just reshaping matrix sizes
            logits_flat = vocab_logits.reshape(B * T, C)

            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(logits_flat, targets_flat)
        return vocab_logits, loss

    # This is autoregressive generation, basically jsut inference
    @torch.no_grad()  # no grad is very helpful in inference as runnign gradient calcualtions take unessesary processing power
    def generate(self, x, max_new_tokens):

        for _ in range(max_new_tokens):

            # Transformer can only handle block_size tokens at once.
            # This is once again because the block_size is the method that is used to splice the text
            x_context = x[:, -BLOCK_SIZE:]

            # forward pass
            logits, _ = self(x_context)

            # Only care about prediction at the final token position
            # this is also where I could add an optional tempreature value, and make no significant change
            # to the general model code
            # Tempreature works by essentially modifying the distributions of the final outputs,
            # ultimately increasing the 'entropy' of the result and therefore
            # making the models outputs more creative
            logits = logits[:, -1, :]

            # [B, vocab_size]

            # Convert logits into probabilities, this is where tempreature can be used
            # also softmax is imporant to keep the entire probability distribution between 0 and 1
            logits /= temperature
            probabilities = F.softmax(logits, dim=-1)

            # smapling next token based on the probabilities (places dice)
            next_token = torch.multinomial(probabilities, num_samples=1)

            # [B, 1]

            # Aadd token to sequence so that the model iterately increases the context
            x = torch.cat((x, next_token), dim=1)

        return x


# the estimate / validation loss (basic concept for gradient descent)
# this helps give an idea as to how good the model is converging onto the correct behaviors
@torch.no_grad()
def estimate_loss(model):

    model.eval()

    results = {}

    for split_name in ["train", "val"]:

        # loss is initally 0, but changes after being run
        losses = torch.zeros(eval_iters)

        for i in range(eval_iters):

            x, y = get_batch(split_name)

            _, loss = model(x, y)

            losses[i] = loss.item()

        results[split_name] = losses.mean().item()

    model.train()

    return results


# now creating the actual model and stuff is pretty straightforward, however an interesting thing
# to note is that the model itself is small enough to be loaded straight onto the GPU memory
# of my laptop, so I noticed that is was very very fast (for not using parallel self-attention computation)
model = SmallLanguageModel().to(device)

num_parameters = sum(p.numel() for p in model.parameters())

print(f"Model parameters: {num_parameters:,}")


# optimizing the model to prevent overfitting
# this is a VERY essential point in making the model perform well
# as without the gold-standard Adam-Weight decay optimizer, the model quickly balloons certain
# weights to be too large and ends up memorizing everything
# AdamW is also a very interesting algorithm, see, AdamW only keeps in 2 pieces of previous data
# which is the momentum (first moment of the weight) (essentially the average direction the weight
# has been moving)
# and then the RMSProp (second moment of the weight) (which finds the magnitude of recent updates)
# This is actually very similar to how algorithms such as PID and even most filters work
# If an LLM is basically a really efficient compression engine, then optimization through AdamW
# is going to be the filter that helps keep the compression engine as effective as possible
# This is very similar to physics through velocity and acceleration
# AdamW will keep the velocity if it has been consistently in a certain direction
# AdamW will dampen the acceleration if the weight is erratic, and amplify the acceleration if it is not moving
# AdamW is basically the great equalizer and judge
# highly highly cool
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


# this is training loop

for iteration in range(max_iters):

    # don't want to constantly evaluate, so only done every eval_interval ammount (for sexiness)
    if iteration % eval_interval == 0:

        losses = estimate_loss(model)

        print(
            f"step {iteration:5d} | "
            f"train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f}"
        )

    # smapling the batch

    x, y = get_batch("train")

    # forward pass

    logits, loss = model(x, y)

    # backprop
    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()


# generate text 
model.eval()



Model parameters: 10,806,250
step     0 | train loss 4.8399 | val loss 4.8817
step   200 | train loss 2.4396 | val loss 2.4222
step   400 | train loss 2.3251 | val loss 2.2679
step   600 | train loss 2.0668 | val loss 2.1086


In [ ]:
# since the model needs data to run, assigning first start token is most important
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=10000)
hashes = generated[0].tolist()
# for i in range(len(hashes)):
#     hashes[i] += (random.random())
#     hashes[i] = int(round(hashes[i]))
    
generated_text = decode(hashes)
print("\n")
print("=" * 60)
print("GENERATED TEXT")
print("=" * 60)
print(generated_text)



GENERATED TEXT

```

The                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              